In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END, add_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Annotated, List, Sequence
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import ServerlessSpec, Pinecone
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
    RunnableLambda,
)
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import GoogleGenerativeAIEmbeddings
import os
from langgraph.prebuilt import ToolNode, tools_condition
from langfuse import get_client
from langfuse.langchain import CallbackHandler

d:\ProdRAG\prodRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [ ]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index_name = "practice-rag"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        serverless=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

In [ ]:
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0.5)

In [ ]:
class RAGState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]

In [6]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_prompt = PromptTemplate.from_template(
    template="""You are a helpful assistant.
Use only the provided context to answer the question.
If the answer is not present in the context, say: "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:"""
)

In [8]:
file_path = r"D:\ProdRAG\prodRAG\Blockchain_Course_Proposal.pdf"
loader = PyPDFLoader(file_path)
documents = loader.load()

In [9]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
texts = text_splitter.split_documents(documents)

In [ ]:
embedder = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview", output_dimensionality=384
)

In [ ]:
vectorstore = PineconeVectorStore.from_documents(texts, embedder, index_name=index_name)

In [12]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [ ]:
@tool
def rag_tool(query: str) -> str:
    """A tool that retrieves relevant information from the PDF based on a user query.
    Use this tool when the user asks factual/conceptual questions that might be answered from the stored documents.
    """

    # Retrieve similar chunks from the vectorstore
    docs = retriever.invoke(query)
    context = format_docs(docs)

    return context

In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chat_rag(state: RAGState):
    # Create a system message that instructs the LLM to use tool results
    system_prompt = """You are a helpful assistant. When you use the rag_tool, carefully read the context it returns and base your answer ONLY on that context.
If the context doesn't contain the answer, say 'I don't know based on the provided context.' Do NOT use your general knowledge if the tool provides information."""

    # Prepare messages with system context
    messages_with_system = [SystemMessage(content=system_prompt)] + state["messages"]

    response = llm_with_tools.invoke(messages_with_system)
    return {"messages": [response]}

In [ ]:
tool_node = ToolNode(tools)

In [ ]:
workflow = StateGraph(RAGState)
workflow.add_node("chat_rag", chat_rag)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "chat_rag")
workflow.add_conditional_edges(
    "chat_rag", tools_condition, {"tools": "tools", END: END}
)
workflow.add_edge("tools", "chat_rag")

app = workflow.compile()

In [19]:
langfuse = get_client()
langfuse_handler = CallbackHandler()

In [ ]:
result = app.invoke(
    {"messages": [HumanMessage(content=("What is introduction?"))]},
    # config={"callbacks": [langfuse_handler]}
)

In [26]:
print(result)

{'messages': [HumanMessage(content='What is introduction?', additional_kwargs={}, response_metadata={}, id='cb6dfc63-f0ee-4213-b7c1-d30a95d4879e'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'k6tzzr0s0', 'function': {'arguments': '{"query":"introduction"}', 'name': 'rag_tool'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 338, 'total_tokens': 354, 'completion_time': 0.031791822, 'completion_tokens_details': None, 'prompt_time': 0.033676982, 'prompt_tokens_details': None, 'queue_time': 0.159317936, 'total_time': 0.065468804}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d52e4-851d-7353-a4d6-519456b675cf-0', tool_calls=[{'name': 'rag_tool', 'args': {'query': 'introduction'}, 'id': 'k6tzzr0s0', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tok

In [ ]:
{
    "messages": [
        HumanMessage(
            content="What is module 1 and module 2?",
            additional_kwargs={},
            response_metadata={},
            id="be69adba-46ca-40ba-88ba-d332544544d9",
        ),
        AIMessage(
            content="I don't know based on the provided context.",
            additional_kwargs={},
            response_metadata={
                "token_usage": {
                    "completion_tokens": 11,
                    "prompt_tokens": 276,
                    "total_tokens": 287,
                    "completion_time": 0.006952024,
                    "completion_tokens_details": None,
                    "prompt_time": 0.032837122,
                    "prompt_tokens_details": None,
                    "queue_time": 0.187817126,
                    "total_time": 0.039789146,
                },
                "model_name": "llama-3.3-70b-versatile",
                "system_fingerprint": "fp_45180df409",
                "service_tier": "on_demand",
                "finish_reason": "stop",
                "logprobs": None,
                "model_provider": "groq",
            },
            id="lc_run--019d5437-1aa1-7a72-85c1-c88c740e7e96-0",
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                "input_tokens": 276,
                "output_tokens": 11,
                "total_tokens": 287,
            },
        ),
    ],
    "file_path": "D:\\ProdRAG\\prodRAG\\Blockchain_Course_Proposal.pdf",
}

In [ ]:
{
    "messages": [
        HumanMessage(
            content="What is introduction?",
            additional_kwargs={},
            response_metadata={},
            id="cb6dfc63-f0ee-4213-b7c1-d30a95d4879e",
        ),
        AIMessage(
            content="",
            additional_kwargs={
                "tool_calls": [
                    {
                        "id": "k6tzzr0s0",
                        "function": {
                            "arguments": '{"query":"introduction"}',
                            "name": "rag_tool",
                        },
                        "type": "function",
                    }
                ]
            },
            response_metadata={
                "token_usage": {
                    "completion_tokens": 16,
                    "prompt_tokens": 338,
                    "total_tokens": 354,
                    "completion_time": 0.031791822,
                    "completion_tokens_details": None,
                    "prompt_time": 0.033676982,
                    "prompt_tokens_details": None,
                    "queue_time": 0.159317936,
                    "total_time": 0.065468804,
                },
                "model_name": "llama-3.1-8b-instant",
                "system_fingerprint": "fp_7ccc667439",
                "service_tier": "on_demand",
                "finish_reason": "tool_calls",
                "logprobs": None,
                "model_provider": "groq",
            },
            id="lc_run--019d52e4-851d-7353-a4d6-519456b675cf-0",
            tool_calls=[
                {
                    "name": "rag_tool",
                    "args": {"query": "introduction"},
                    "id": "k6tzzr0s0",
                    "type": "tool_call",
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                "input_tokens": 338,
                "output_tokens": 16,
                "total_tokens": 354,
            },
        ),
        ToolMessage(
            content="Course Proposal: Blockchain Technology and Its Applications in Modern \nComputing \n1. Introduction \nWith the rapid advancement of decentralized technologies, Blockchain has emerged as a \ntransformative force across industries. From finance to healthcare and IoT, blockchain \nensures transparency, security, and data integrity without relying on centralized control. \nThis course aims to provide students with comprehensive theoretical and practical\n\nCourse Proposal: Blockchain Technology and Its Applications in Modern \nComputing \n1. Introduction \nWith the rapid advancement of decentralized technologies, Blockchain has emerged as a \ntransformative force across industries. From finance to healthcare and IoT, blockchain \nensures transparency, security, and data integrity without relying on centralized control. \nThis course aims to provide students with comprehensive theoretical and practical\n\nCourse Proposal: Blockchain Technology and Its Applications in Modern \nComputing \n1. Introduction \nWith the rapid advancement of decentralized technologies, Blockchain has emerged as a \ntransformative force across industries. From finance to healthcare and IoT, blockchain \nensures transparency, security, and data integrity without relying on centralized control. \nThis course aims to provide students with comprehensive theoretical and practical",
            name="rag_tool",
            id="d16627fe-22bf-4c49-b96a-6b46b74389e2",
            tool_call_id="k6tzzr0s0",
        ),
        AIMessage(
            content="The introduction to the course is about blockchain technology and its applications in modern computing, highlighting its transformative force across various industries and its ability to ensure transparency, security, and data integrity.",
            additional_kwargs={},
            response_metadata={
                "token_usage": {
                    "completion_tokens": 37,
                    "prompt_tokens": 588,
                    "total_tokens": 625,
                    "completion_time": 0.065506934,
                    "completion_tokens_details": None,
                    "prompt_time": 0.032992804,
                    "prompt_tokens_details": None,
                    "queue_time": 0.160935347,
                    "total_time": 0.098499738,
                },
                "model_name": "llama-3.1-8b-instant",
                "system_fingerprint": "fp_7ccc667439",
                "service_tier": "on_demand",
                "finish_reason": "stop",
                "logprobs": None,
                "model_provider": "groq",
            },
            id="lc_run--019d52e4-980b-7f42-aec7-d756a805f600-0",
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                "input_tokens": 588,
                "output_tokens": 37,
                "total_tokens": 625,
            },
        ),
    ]
}

In [27]:
print(result["messages"][-1].content)

The introduction to the course is about blockchain technology and its applications in modern computing, highlighting its transformative force across various industries and its ability to ensure transparency, security, and data integrity.
